# CS781 Phase 5 - Legal Statute Identification - notebook `3`

**Base design:** document text in Phase-1 tokenization -> **the whole document up to 2,048 tokens** (4 chunks of 512;
longer documents: 4 chunks spread evenly over it) -> Phase-1 backbone -> mean-pooled chunk vectors + relative chunk
position (16 sinusoids) -> **attention pooling** -> **one sigmoid per statute** (binary relevance), BCE loss; 1 epoch over
train.json, batches of 8 documents; checkpoint chosen on a fixed 3,000-document dev subset (macro-F1 at the best global
threshold), evaluated every 25% of the steps; the final report uses all of dev with three decision rules (0.5, best
global threshold, per-statute thresholds + an honest cross-fitted estimate); every document gets at least one statute.
Training documents that also occur in dev/test are removed.
**Techniques** (T, every model): LoRMA (compulsory), AdaLoRA, QLoRA, adapters, prefix tuning, partial FT.
**Study** (S, models B and D, LoRMA, the same 12,000-document subset): which part of the document, head (label-wise
attention, chunk-level predictions + max, mean pooling), loss (BCE + positive weights, ASL, focal), statute co-occurrence.
**Extras** (X, run last): full FT, prompt tuning. Knowledge distillation is optional (code ready, not scheduled).

| ID | GPU | run | technique | change vs base design | group | estimate |
|---|---|---|---|---|---|---|
| D03 | GPU0 | `D03_qlora` | QLoRA (NF4 + LoRA r16) | - | T | ~175 min |
| E02 | GPU0 | `E02_adalora` | AdaLoRA r12->8 | - | T | ~61 min |
| E05 | GPU0 | `E05_prefix` | Prefix tuning (20) | - | T | ~52 min |
| E06 | GPU0 | `E06_partial` | Partial FT (top 4) | - | T | ~33 min |
| D08 | GPU0 | `D08_prompt` | Prompt tuning (20) | - | X | ~119 min |
| E07 | GPU0 | `E07_full` | Full FT | - | X | ~61 min |
| D02 | GPU1 | `D02_adalora` | AdaLoRA r12->8 | - | T | ~138 min |
| E03 | GPU1 | `E03_qlora` | QLoRA (NF4 + LoRA r16) | - | T | ~77 min |
| E01 | GPU1 | `E01_lorma` | LoRMA r16 | - | T | ~55 min |
| E04 | GPU1 | `E04_adapter` | Houlsby adapter | lr 1e-4 | T | ~47 min |
| D07 | GPU1 | `D07_full` | Full FT | - | X | ~138 min |
| E08 | GPU1 | `E08_prompt` | Prompt tuning (20) | - | X | ~52 min |

**Inputs:** dataset `cs781-p5-code`, `cs781-legal-data` (tokenizer), Phase-1 checkpoint(s) of model D + E.
Internet ON, secret `HF_TOKEN`, GPU **T4 x2**. Save Version -> Save & Run All.
Expected: ~8.4 h of runs (GPU0 8.3 h, GPU1 8.4 h) after ~5 min of tokenization and a
~5 min smoke test. Each run has a time limit (it then stops, is evaluated on dev and kept); no run starts after
11.3 h, so the notebook always finishes inside Kaggle's 12-hour limit. A run that fails the smoke test is dropped
(its error is in the report); the others continue.
**Outputs:** `submissions/<run>.zip` (ready to submit), `models/<run>.pt`, `bundle_p5_3.zip` (dev/test probabilities,
attention weights, logs; input of `cs781-p5-final`), `REPORT_p5_3.txt`.

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading, zipfile
NB_START = time.time()
W = "/kaggle/working"
WORK, P5DATA, RUNS, MODELS = f"{W}/p5code", "/tmp/p5data", f"{W}/runs", f"{W}/models"
CACHE = "/tmp/p5_cache.npz"                        # tokenized data (outside /kaggle/working: not part of the output)
for d in (RUNS, MODELS, f"{W}/logs", f"{W}/submissions", P5DATA):
    os.makedirs(d, exist_ok=True)

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("lsi_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", sorted(f for f in os.listdir(WORK) if f.endswith(".py")))

# ---- Phase-5 data from Hugging Face (Phase_5_LSI/Train/{train,dev}.json, Phase_5_LSI/Test/test.json)
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
files = [f for f in list_repo_files(REPO, repo_type="dataset", token=tok_) if f.startswith("Phase_5")]
print("HF files:", files)
for f in files:
    dst = os.path.join(P5DATA, f)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), dst)

# ---- Phase-1 checkpoints (final.pt per model letter; continual-pretrained *_cpt preferred)
CKPT = {}
ALL_PT = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True))
print("checkpoint files found under /kaggle/input:", *ALL_PT, sep="\n  ")
for p in ALL_PT:
    if os.path.basename(p) != "final.pt" or "smoke" in p.lower():
        continue
    m = re.search(r"run[_-]([a-e])(?:[_/-]|$)", p, re.I) or re.search(r"model[_-]?([a-e])(?:[_/-]|$)", p, re.I)
    if not m:
        continue
    k = m.group(1).lower()
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
CKPT.update(CKPT_OVERRIDE if "CKPT_OVERRIDE" in globals() else {})   # e.g. CKPT_OVERRIDE = {"c": "/kaggle/input/.../final.pt"}
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

# ---- unit tests, then tokenize every split once (shared by all runs; several CPU processes, a few minutes)
r = subprocess.run("python tests_lsi.py", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"
import lsi_data
t0 = time.time()
INFO = lsi_data.build_cache(P5DATA, TOK_ROOT, CACHE)
print(f"cache built in {time.time() - t0:.0f}s")
if "test" not in INFO:
    print("WARNING: no test.json found - runs will train and save models, but write no submission files")

code: ['adapt.py', 'attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'gated_deltanet.py', 'kv_cache.py', 'layers.py', 'lsi_data.py', 'lsi_eval.py', 'lsi_model.py', 'lsi_train.py', 'models.py', 'ner_data.py', 'ner_model.py', 'p1_loader.py', 'peft.py', 'tests_lsi.py']
HF files: ['Phase_5_LSI/Test/test.json', 'Phase_5_LSI/Train/dev.json', 'Phase_5_LSI/Train/train.json']


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

checkpoint files found under /kaggle/input:
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-e/pytorch/phase-1/1/run_e_small_gpt_55m/best.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-e/pytorch/phase-1/1/run_e_small_gpt_55m/final.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-e/pytorch/phase-1/1/run_e_small_gpt_55m/latest.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/best.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/final.pt
  /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/latest.pt
  model d: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/final.pt
  model e: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-e/pytorch/phase-1/1/run_e_small_gpt_55m/final.pt
name, memory.total [MiB]
Tesla T4, 15360 MiB
T

In [2]:
NB_HOURS = 11.5                                    # Kaggle stops a session at 12 h; no run starts after this

def cmd(e, out, extra, model_dir):
    return (f"python lsi_train.py --ckpt {CKPT[e['model']]} --tok_root {TOK_ROOT} --data_dir {P5DATA} --cache {CACHE} "
            f"--out {out} --model_dir {model_dir} {e['args']} {extra}")

def run(tag, c, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    with open(f"{W}/logs/{tag}.txt", "w") as f:
        p = subprocess.run(c, shell=True, cwd=WORK, env=env, stdout=f, stderr=subprocess.STDOUT)
    tail = open(f"{W}/logs/{tag}.txt").read().strip().splitlines()[-2:]
    print(f"[GPU{gpu}] {tag}: exit {p.returncode} | " + " / ".join(tail)[-300:], flush=True)
    return p.returncode

def run_queues(queues, smoke=False):
    """queues: one list of experiments per GPU, run in parallel. smoke: a few documents, a few steps."""
    status = {}
    def worker(gpu, jobs):
        for e in jobs:
            t = ("smoke_" if smoke else "") + e["tag"]
            if smoke:
                extra = "--max_docs 24 --batch_docs 4 --eval_every 0.5" + ("" if e is jobs[0] else " --no_test")
            else:
                left = NB_HOURS * 60 - (time.time() - NB_START) / 60
                if left < 20:
                    print(f"[GPU{gpu}] {t}: SKIPPED (notebook time budget used up)", flush=True)
                    status[t] = "skipped"
                    continue
                extra = f"--max_minutes {min(e['max_min'], left - 15):.0f}"
            try:
                status[t] = run(t, cmd(e, f"{RUNS}/{t}", extra, f"{W}/smoke_models" if smoke else MODELS), gpu)
            except Exception as ex:                       # e.g. a missing checkpoint: report it, keep the queue going
                print(f"[GPU{gpu}] {t}: NOT STARTED - {ex!r}", flush=True)
                open(f"{W}/logs/{t}.txt", "w").write(f"not started: {ex!r}")
                status[t] = "error"
    th = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues) if q]
    t0 = time.time()
    for t in th: t.start()
    for t in th: t.join()
    print(f"{'smoke' if smoke else 'runs'} done in {(time.time() - t0) / 60:.1f} min:", status)
    return status

def smoke(queues):
    """Every experiment once on 24 documents. A failing experiment is dropped (its error is printed and reported)
    instead of stopping the notebook; the notebook stops only if every experiment fails."""
    st = run_queues(queues, smoke=True)
    bad = [k for k, v in st.items() if v != 0]
    for k in bad:
        print(f"===== SMOKE FAILED, dropped: {k}\n" + open(f"{W}/logs/{k}.txt").read()[-3000:])
    assert len(bad) < len(st), "every experiment failed the smoke test"
    for k in st:
        shutil.rmtree(f"{RUNS}/{k}", ignore_errors=True)
    shutil.rmtree(f"{W}/smoke_models", ignore_errors=True)
    print("SMOKE OK" if not bad else f"SMOKE OK except {len(bad)} dropped: {bad}")
    return [[e for e in q if "smoke_" + e["tag"] not in bad] for q in queues]

## Experiments (one queue per GPU)

In [3]:
NAME = '3'
QUEUES = [[{'id': 'D03',
   'tag': 'D03_qlora',
   'model': 'd',
   'tech': 'QLoRA (NF4 + LoRA r16)',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method qlora --rank 16 --alpha 32',
   'max_min': 285},
  {'id': 'E02',
   'tag': 'E02_adalora',
   'model': 'e',
   'tech': 'AdaLoRA r12->8',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method adalora --rank 12 --alpha 24 --target_rank 8',
   'max_min': 102},
  {'id': 'E05',
   'tag': 'E05_prefix',
   'model': 'e',
   'tech': 'Prefix tuning (20)',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method prefix --n_prompt 20',
   'max_min': 89},
  {'id': 'E06',
   'tag': 'E06_partial',
   'model': 'e',
   'tech': 'Partial FT (top 4)',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method partial --top_k 4',
   'max_min': 58},
  {'id': 'D08',
   'tag': 'D08_prompt',
   'model': 'd',
   'tech': 'Prompt tuning (20)',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method prompt --n_prompt 20',
   'max_min': 195},
  {'id': 'E07',
   'tag': 'E07_full',
   'model': 'e',
   'tech': 'Full FT',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method full',
   'max_min': 102}],
 [{'id': 'D02',
   'tag': 'D02_adalora',
   'model': 'd',
   'tech': 'AdaLoRA r12->8',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method adalora --rank 12 --alpha 24 --target_rank 8',
   'max_min': 225},
  {'id': 'E03',
   'tag': 'E03_qlora',
   'model': 'e',
   'tech': 'QLoRA (NF4 + LoRA r16)',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method qlora --rank 16 --alpha 32',
   'max_min': 128},
  {'id': 'E01',
   'tag': 'E01_lorma',
   'model': 'e',
   'tech': 'LoRMA r16',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method lorma --rank 16 --alpha 32 --lr 2e-4',
   'max_min': 93},
  {'id': 'E04',
   'tag': 'E04_adapter',
   'model': 'e',
   'tech': 'Houlsby adapter',
   'change': 'lr 1e-4',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method adapter --bottleneck 64 --lr 1e-4',
   'max_min': 80},
  {'id': 'D07',
   'tag': 'D07_full',
   'model': 'd',
   'tech': 'Full FT',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method full',
   'max_min': 225},
  {'id': 'E08',
   'tag': 'E08_prompt',
   'model': 'e',
   'tech': 'Prompt tuning (20)',
   'change': '-',
   'args': '--select spread --n_chunks 4 --head attn --loss bce --method prompt --n_prompt 20',
   'max_min': 89}]]
ALL_QUEUES = [list(q) for q in QUEUES]                    # the full plan (the report lists dropped runs too)

## 1. Smoke test

In [4]:
QUEUES = smoke(QUEUES)

[GPU1] smoke_D02_adalora: exit 0 | [20:10:43] TEST: 24 documents, 98.12 statutes each (threshold 0.15) -> submission.zip / [20:10:43] DONE
[GPU0] smoke_D03_qlora: exit 0 | [20:10:43] TEST: 24 documents, 99.33 statutes each (threshold 0.17) -> submission.zip / [20:10:43] DONE
[GPU0] smoke_E02_adalora: exit 0 | [20:11:06] TEST: no test.json - no submission written / [20:11:06] DONE
[GPU1] smoke_E03_qlora: exit 0 | [20:11:08] TEST: no test.json - no submission written / [20:11:08] DONE
[GPU0] smoke_E05_prefix: exit 0 | [20:11:15] TEST: no test.json - no submission written / [20:11:15] DONE
[GPU1] smoke_E01_lorma: exit 0 | [20:11:18] TEST: no test.json - no submission written / [20:11:18] DONE
[GPU0] smoke_E06_partial: exit 0 | [20:11:24] TEST: no test.json - no submission written / [20:11:24] DONE
[GPU1] smoke_E04_adapter: exit 0 | [20:11:27] TEST: no test.json - no submission written / [20:11:27] DONE
[GPU0] smoke_D08_prompt: exit 0 | [20:11:42] TEST: no test.json - no submission written

## 2. Runs

In [5]:
status = run_queues(QUEUES)

[GPU0] D03_qlora: exit 0 | [22:48:03] TEST: 13019 documents, 6.73 statutes each (threshold 0.13) -> submission.zip / [22:48:03] DONE
[GPU1] D02_adalora: exit 0 | [22:55:05] TEST: 13019 documents, 7.33 statutes each (threshold 0.12) -> submission.zip / [22:55:05] DONE
[GPU0] E02_adalora: exit 0 | [00:01:45] TEST: 13019 documents, 6.55 statutes each (threshold 0.14) -> submission.zip / [00:01:45] DONE
[GPU1] E03_qlora: exit 0 | [00:04:10] TEST: 13019 documents, 6.69 statutes each (threshold 0.13) -> submission.zip / [00:04:10] DONE
[GPU0] E05_prefix: exit 0 | [00:57:23] TEST: 13019 documents, 7.54 statutes each (threshold 0.12) -> submission.zip / [00:57:23] DONE
[GPU1] E01_lorma: exit 0 | [01:09:18] TEST: 13019 documents, 6.85 statutes each (threshold 0.13) -> submission.zip / [01:09:18] DONE
[GPU0] E06_partial: exit 0 | [01:34:36] TEST: 13019 documents, 6.33 statutes each (threshold 0.14) -> submission.zip / [01:34:36] DONE
[GPU1] E04_adapter: exit 0 | [02:02:02] TEST: 13019 documents,

## 3. Report

In [6]:
def log_tail(name, n=2):
    p = f"{W}/logs/{name}.txt"
    return " / ".join(open(p).read().strip().splitlines()[-n:])[-250:] if os.path.exists(p) else ""

def summarize(name, queues):
    exps = [e for q in queues for e in q]
    L = [f"=== REPORT {name} ===",
         "ID  | run                     | technique            | change                         | mF1@thr     | mF1 per-label (cross-fit) | micro  | subset | lab/doc | steps     | trainable | min  | tok/s  | mem   | ms/doc"]
    for e in exps:
        p = f"{RUNS}/{e['tag']}/summary.json"
        if not os.path.exists(p):
            why = "SKIPPED/DROPPED" if not os.path.exists(f"{W}/logs/{e['tag']}.txt") else "FAILED"
            L.append(f"{e['id']} | {e['tag']:23s} | {why}: {log_tail(e['tag']) or log_tail('smoke_' + e['tag'])}")
            continue
        s = json.load(open(p)); dv = s["dev"]; g = dv["global"]
        L.append(f"{e['id']} | {e['tag']:23s} | {e['tech'][:20]:20s} | {e['change'][:30]:30s} | {g['macro_f1']:.4f}@{dv['global_threshold']:.2f} | "
                 f"{dv['per_label']['macro_f1']:.4f} ({dv['crossfit']['per_label']:.4f})          | {g['micro_f1']:.4f} | "
                 f"{g['subset_acc']:.4f} | {g['pred_per_doc']:5.2f}   | {s.get('best_step', 0):4d}/{s.get('steps_planned', 0):4d} | "
                 f"{s['params_trainable'] / 1e6:8.2f}M | {s['train_minutes']:4.0f} | {s.get('tok_s') or 0:6,.0f} | "
                 f"{s.get('peak_mem_gb') or 0:4.1f}G | {s.get('infer_ms_per_doc', 0):6.1f}")
        if os.path.exists(f"{RUNS}/{e['tag']}/submission.zip"):
            shutil.copy(f"{RUNS}/{e['tag']}/submission.zip", f"{W}/submissions/{e['tag']}.zip")
    bundle = f"{W}/bundle_p5_{name}.zip"              # everything except model weights (those stay in models/)
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for e in exps:
            d = f"{RUNS}/{e['tag']}"
            if os.path.isdir(d):
                for f in os.listdir(d):
                    z.write(f"{d}/{f}", arcname=f"{e['tag']}/{f}")
            if os.path.exists(f"{W}/logs/{e['tag']}.txt"):
                z.write(f"{W}/logs/{e['tag']}.txt", arcname=f"{e['tag']}/stdout.txt")
    L.append(f"\nbundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB)")
    L.append("models: " + ", ".join(f"{f} ({os.path.getsize(f'{MODELS}/{f}') / 2**20:.0f} MB)" for f in sorted(os.listdir(MODELS))))
    L.append("submission files: " + ", ".join(sorted(os.listdir(f"{W}/submissions"))))
    rep = "\n".join(L)
    open(f"{W}/REPORT_p5_{name}.txt", "w").write(rep)
    print(rep + "\n=== END REPORT ===")

summarize(NAME, ALL_QUEUES)

=== REPORT 3 ===
ID  | run                     | technique            | change                         | mF1@thr     | mF1 per-label (cross-fit) | micro  | subset | lab/doc | steps     | trainable | min  | tok/s  | mem   | ms/doc
D03 | D03_qlora               | QLoRA (NF4 + LoRA r1 | -                              | 0.3573@0.13 | 0.3915 (0.3759)          | 0.4760 | 0.0191 |  6.79   | 5304/5304 |     3.05M |  135 |  4,941 |  5.2G |   53.4
E02 | E02_adalora             | AdaLoRA r12->8       | -                              | 0.3269@0.14 | 0.3656 (0.3480)          | 0.4622 | 0.0125 |  6.60   | 5304/5304 |     1.49M |   63 | 10,717 |  2.5G |   27.2
E05 | E05_prefix              | Prefix tuning (20)   | -                              | 0.3053@0.12 | 0.3453 (0.3288)          | 0.4338 | 0.0061 |  7.57   | 5304/5304 |     0.28M |   46 | 14,717 |  2.2G |   23.9
E06 | E06_partial             | Partial FT (top 4)   | -                              | 0.3472@0.14 | 0.3824 (0.3657)          | 0.478